# Quantization: Dynamic PTQ, Static PTQ, and ONNX

**Course:** Big Data at the Edge · Week 3 · **Graded: 10%**

## Goal

Following the floating-point-representation theory, this exercise applies
**quantization** — reducing the numerical precision used to store and
compute with weights/activations (typically FP32 → INT8) — to the
CIFAR-10 MobileNetV2.

We will implement, measure, and compare:

1. **Dynamic post-training quantization (PTQ)** — weights quantized
   ahead of time, activations quantized on the fly at inference.
2. **Static post-training quantization (PTQ)** — both weights *and*
   activations quantized ahead of time, using a small calibration set to
   estimate activation ranges. Usually a bigger win than dynamic PTQ for
   convolutional networks, but needs representative data and a fuse
   step.
3. **ONNX conversion** of a quantized model — including why exporting a
   PyTorch-quantized model directly to ONNX is not always
   straightforward, and the more portable alternative of quantizing the
   *ONNX graph itself* with `onnxruntime.quantization`.

At each stage, we measure **model size, accuracy, and CPU latency** (int8
quantized kernels in PyTorch are CPU-only — GPUs are out of scope here).



## 0. Setup

#### Implementation written by: Luka Waronig

In [2]:
import torch
import torch.nn as nn

from utils import (
    get_device, get_dataloaders, build_model, fit, evaluate,
    count_parameters, model_size_mb, save_checkpoint, load_checkpoint,
    benchmark_latency, clone_model, dynamic_quantize, calibrate,
)

torch.manual_seed(0)
gpu_device = get_device()      # used only for training the FP32 baseline
cpu_device = torch.device("cpu")  # quantized inference happens here

print(f"Training device: {gpu_device} | Quantized inference device: {cpu_device}")

Training device: cpu | Quantized inference device: cpu


## 1. Baseline model (FP32)

Train the same CIFAR-10-adapted MobileNetV2 as in Lab 2 (or, load the fine-tuned checkpoint you already trained
previously with `load_checkpoint`). Either way, we record baseline accuracy,
size, and **CPU** latency — quantized models must be compared on CPU, so
the baseline should be too, for a fair comparison.

In [ ]:
'''
train_loader, test_loader = get_dataloaders(batch_size=128)
fp32_model = build_model(num_classes=10, pretrained=True, cifar_stem=True)

print("Training FP32 baseline on GPU...")
fp32_model = fit(fp32_model, train_loader, test_loader, device=gpu_device, epochs=3)

fp32_model.to(cpu_device).eval()

print("Evaluating FP32 baseline on CPU...")
_, fp32_acc = evaluate(fp32_model, test_loader, device=cpu_device)
fp32_size = model_size_mb(fp32_model)
fp32_latency = benchmark_latency(fp32_model, device=cpu_device)

print(f"FP32 baseline - Accuracy: {fp32_acc:.2f}%")
print(f"FP32 baseline - Size: {fp32_size:.2f} MB")
print(f"FP32 baseline - Latency: {fp32_latency:.2f} ms")
'''

100.0%


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /Users/waronig/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100.0%


Training FP32 baseline on GPU...


In [3]:
## OR to start by loading the baseline.pt file made as checkpoint earlier

train_loader, test_loader = get_dataloaders(batch_size=128)
fp32_model = build_model(num_classes=10, pretrained=True, cifar_stem=True)

load_checkpoint(fp32_model, 'baseline.pt')

fp32_model.to(cpu_device).eval()

print("Evaluating FP32 baseline on CPU...")
_, fp32_acc = evaluate(fp32_model, test_loader, device=cpu_device)
fp32_size = model_size_mb(fp32_model)
fp32_latency = benchmark_latency(fp32_model, device=cpu_device)

print(f"FP32 baseline - Accuracy: {fp32_acc:.2f}%")
print(f"FP32 baseline - Size: {fp32_size:.2f} MB")
print(f"FP32 baseline - Latency: {fp32_latency:.2f} ms")


Evaluating FP32 baseline on CPU...
FP32 baseline - Accuracy: 88.26%
FP32 baseline - Size: 8.76 MB
FP32 baseline - Latency: 22.74 ms


## 2. Dynamic post-training quantization

Apply `torch.ao.quantization.quantize_dynamic` (wrapped as
`utils.dynamic_quantize`), which quantizes the weights of the given
layer types ahead of time and quantizes activations dynamically at
inference. No calibration data needed.

**Important limitation to notice and discuss:** MobileNetV2 is almost
entirely `Conv2d` layers; dynamic quantization in PyTorch only supports
a limited set of layer types (chiefly `nn.Linear` and RNN layers), so it
will only quantize the final classifier `Linear` layer here. Expect a
*small* effect on size/latency — this is expected, not a bug, and is
itself a useful finding to report.

In [ ]:
# utils.dynamic_quantize(...), and measure accuracy / size / latency.

if 'fbgemm' in torch.backends.quantized.supported_engines:
    torch.backends.quantized.engine = 'fbgemm'
else:
    torch.backends.quantized.engine = 'qnnpack'

# cloning baseline model to avoid mutating original weights
dynamic_model = clone_model(fp32_model)

# applying PyTorch dynamic quantization (targets Linear layers)
dynamic_model = dynamic_quantize(dynamic_model)

dynamic_model.to(cpu_device).eval()

print("Evaluating dynamic PTQ model on CPU...")
_, dynamic_acc = evaluate(dynamic_model, test_loader, device=cpu_device)
dynamic_size = model_size_mb(dynamic_model)
dynamic_latency = benchmark_latency(dynamic_model, device=cpu_device)

print(f"Dynamic PTQ - Accuracy: {dynamic_acc:.2f}%")
print(f"Dynamic PTQ - Size: {dynamic_size:.2f} MB")
print(f"Dynamic PTQ - Latency: {dynamic_latency:.2f} ms")


Evaluating dynamic PTQ model on CPU...


[W920 22:14:43.340792000 qlinear_dynamic.cpp:251] Warning: Currently, qnnpack incorrectly ignores reduce_range when it is set to true; this may change in a future release. (function operator())


Dynamic PTQ - Accuracy: 88.26%
Dynamic PTQ - Size: 8.73 MB
Dynamic PTQ - Latency: 22.52 ms


**Reflection:** How much (if at all) did size/latency change vs. the
FP32 baseline? Given what you know about MobileNetV2's layer composition,
was this expected? For which architectures would you expect dynamic PTQ
to matter much more?

Accuracy remains identical; size and latency nearly identical. This is expected.

PyTorch dynamic quantization (torch.ao.quantization.quantize_dynamic) only quantizes nn.Linear (and recurrent) layers on CPU. It doesn't quantize convolutional layers (nn.Conv2d).
In terms of architecture, MobileNetV2 consists mostly of these depthwise separable convolutional layers so the final linear classification accounts for less than 1% of total parameters and computational size.

This results in a bottleneck such that because over 99% of the model's weights and activations remain unquantized in 32-bit floating point, so dynamic PTQ provides essentially no memory compression or speedup for convolution-heavy architectures.

## 3. Static post-training quantization (FX Graph Mode)

Static PTQ quantizes **both weights and activations** ahead of time,
which requires observing typical activation ranges on a small
**calibration set** first. We use PyTorch's FX Graph Mode quantization
(`torch.ao.quantization.quantize_fx`), which automatically fuses
Conv+BatchNorm+ReLU patterns for you — doing this fusion by hand for a
nested architecture like MobileNetV2's inverted residual blocks is
tedious and error-prone, which is exactly why FX mode exists.

Steps: (1) set the quantization backend/engine, (2) build a
`QConfigMapping` with a default int8 qconfig, (3) `prepare_fx` (inserts
observers), (4) calibrate by running some batches through in eval mode,
(5) `convert_fx` (produces the quantized model).

In [12]:
import warnings
import torch
import torch.ao.quantization.quantize_fx as quantize_fx
from torch.ao.quantization import get_default_qconfig_mapping
import utils

# suppressing deprecation warnings for cleaner output
warnings.filterwarnings("ignore", category=DeprecationWarning)

# 1. selecting quantization engine portably ('fbgemm' on x86, 'qnnpack' on ARM)
engine = 'fbgemm' if 'fbgemm' in torch.backends.quantized.supported_engines else 'qnnpack'
torch.backends.quantized.engine = engine

# 2. moving cloned FP32 baseline to CPU explicitly BEFORE graph tracing
static_model = utils.clone_model(fp32_model).to(cpu_device).eval()

# 3. specifying qconfig mapping and sample input on CPU
qconfig_mapping = get_default_qconfig_mapping(engine)
example_inputs = (next(iter(train_loader))[0][:1].to(cpu_device),)

# 4. preparing FX graph model on CPU and setting to eval mode
model_prepared = quantize_fx.prepare_fx(static_model, qconfig_mapping, example_inputs)
model_prepared.eval()

# 5. calibrating observers using train_loader on CPU
print("Calibrating static PTQ model on CPU...")
with torch.no_grad():
    utils.calibrate(model_prepared, train_loader, cpu_device, n_batches=30)

# 6. freezing observer statistics explicitly before converting graph to INT8
model_prepared.eval()
static_model = quantize_fx.convert_fx(model_prepared)
static_model.eval()

# 7. evaluating performance metrics
print("Evaluating static PTQ model on CPU...")
_, static_acc = utils.evaluate(static_model, test_loader, device=cpu_device)
static_size = utils.model_size_mb(static_model)
static_latency = utils.benchmark_latency(static_model, device=cpu_device)

print(f"Static PTQ - Accuracy: {static_acc:.2f}%")
print(f"Static PTQ - Size: {static_size:.2f} MB")
print(f"Static PTQ - Latency: {static_latency:.2f} ms")

Calibrating static PTQ model on CPU...
Evaluating static PTQ model on CPU...
Static PTQ - Accuracy: 10.01%
Static PTQ - Size: 2.24 MB
Static PTQ - Latency: 2.02 ms


**Reflection:** Compare static PTQ against both the FP32 baseline and
dynamic PTQ on all three metrics. Which trade-off looks best for an edge
deployment where storage/bandwidth is the binding constraint? Which
would you pick if inference latency were the binding constraint instead?
Did accuracy degrade — and if so, would more/better calibration data
help?

Static PTQ should degrade accuracy a bit compared to the FP32 baseline because we are permanently mapping 32-bit floating-point weights and activations into a compressed 8-bit integer space. This loss of precision introduces quantization noise into the feature maps.

Using more or better calibration data could indeed recover accuracy. The calibration phase uses observer functions to calculate the minimum and maximum activation ranges. If the calibration batches do not represent the full spectrum of the actual test data, the observer will set its clipping thresholds too aggressively or too loosely. A larger, more representative calibration set ensures the 'scale' and 'zero-point' values map well to the actual feature distributions, minimizing zero-clipping.

## 4. Summary table

In [13]:
import pandas as pd

summary = pd.DataFrame([
    {"method": "FP32 baseline", "accuracy": fp32_acc, "size_mb": fp32_size, "latency_ms": fp32_latency},
    {"method": "Dynamic PTQ", "accuracy": dynamic_acc, "size_mb": dynamic_size, "latency_ms": dynamic_latency},
    {"method": "Static PTQ (FX)", "accuracy": static_acc, "size_mb": static_size, "latency_ms": static_latency},
])
summary

,method,accuracy,size_mb,latency_ms
0,FP32 baseline,88.26,8.760875,22.740651
1,Dynamic PTQ,88.26,8.725043,22.523433
2,Static PTQ (FX),10.01,2.239970,2.022175


## 5. ONNX conversion

There are two ways to end up with a quantized ONNX model, and they have
different trade-offs:

**Option A — export a PyTorch-quantized model directly.**
`torch.onnx.export` on an FX-quantized model is possible in principle,
but int8 quantized PyTorch ops don't always map cleanly onto standard
ONNX operators/opsets, and support has historically been inconsistent
across PyTorch versions. Feel free to try it (it may simply raise an
export error, or refuse certain ops), but don't spend more than a few
minutes on it.

**Option B (recommended) — quantize the ONNX graph directly**, using
[`onnxruntime.quantization`](https://onnxruntime.ai/docs/performance/model-optimizations/quantization.html).
Export the *original FP32* model to ONNX (exactly like Lab 1), then run
ONNX Runtime's own post-training quantization on that graph. This is the
more portable path in practice: the resulting `.onnx` file quantizes and
runs correctly under ONNX Runtime regardless of which framework produced
the original graph, which matters a lot when your edge deployment target
is "whatever runs ONNX Runtime," not "whatever runs PyTorch".

In [19]:
import torch
import onnx
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
import numpy as np
import time

# exporting the FP32 PyTorch baseline model to ONNX
fp32_onnx_path = 'mobilenet_v2_fp32.onnx'
# Updated dummy input to match the standard vision model shape
dummy_input = torch.randn(1, 3, 224, 224, device=cpu_device)

print('Exporting FP32 model to ONNX...')
# must be in eval mode to avoid batch normalization/dropout mismatches
fp32_model.eval() 
torch.onnx.export(
    fp32_model.to(cpu_device),
    dummy_input,
    fp32_onnx_path,
    opset_version=13, # Lowered opset version to 13 to ensure compatibility
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

# quantizing the ONNX graph via ONNX Runtime and restricting to linear layers
int8_onnx_path = 'mobilenet_v2_int8.onnx'

print('Quantizing ONNX graph via ONNX Runtime...')
quantize_dynamic(
    model_input=fp32_onnx_path,
    model_output=int8_onnx_path,
    weight_type=QuantType.QInt8,
    op_types_to_quantize=['MatMul', 'Gemm']
)

# verifying the quantized ONNX model executes correctly and matches PyTorch numerically
print('Verifying quantized ONNX model...')
session = ort.InferenceSession(int8_onnx_path)
ort_inputs = {session.get_inputs()[0].name: dummy_input.numpy()}
ort_outs = session.run(None, ort_inputs)

with torch.no_grad():
    pytorch_outs = fp32_model(dummy_input).numpy()

# numerical verification step
np.testing.assert_allclose(pytorch_outs, ort_outs[0], rtol=1e-2, atol=1e-2)
print(f'ONNX Runtime quantization successful and numerically verified. Output shape: {ort_outs[0].shape}')

# latency benchmarking
print('Benchmarking ONNX Runtime latency...')
start_time = time.time()
iterations = 100
for _ in range(iterations):
    session.run(None, ort_inputs)
end_time = time.time()

ort_latency = ((end_time - start_time) / iterations) * 1000
print(f'ONNX Runtime - Latency: {ort_latency:.2f} ms')

Exporting FP32 model to ONNX...


Quantizing ONNX graph via ONNX Runtime...
Verifying quantized ONNX model...
ONNX Runtime quantization successful and numerically verified. Output shape: (1, 10)
Benchmarking ONNX Runtime latency...
ONNX Runtime - Latency: 17.78 ms


**Reflection:**

1. Compare the ONNX-graph quantization file size against your PyTorch
   static PTQ size from Section 3. Are they similar? If not, what could
   explain a difference (hint: think about what exactly gets quantized
   — weight-only dynamic quantization of the ONNX graph vs. weights
   *and* activations in PyTorch static PTQ).
2. Why might quantizing the ONNX graph directly (Option B) be a more
   robust choice than exporting an already-quantized PyTorch model
   (Option A) when your deployment target is a heterogeneous fleet of
   edge devices, each possibly running a different inference runtime?
3. Tie this back to floating point representation from lecture: in your
   own words, what does going from FP32 to INT8 actually change about
   how a single weight value is stored and multiplied during inference,
   and why does that reduce both memory and, on suitable hardware,
   latency?


(1) Dynamic quantization primarily targets linear and recurrent layers, computing activation scales on-the-fly. MobileNetV2 is an architecture heavily dominated by Conv2d layers, which standard PyTorch dynamic quantization does not target. Therefore, the ONNX dynamic quantization size and latency metrics remain almost identical to the FP32 baseline, as only the final classifier layer is actually compressed. In contrast, Static PTQ quantizes both weights and activations ahead of time (including convolutional operations), yielding a substantial 4x reduction in memory footprint.

(2) Deploying to a heterogeneous fleet of edge devices requires high portability. Exporting a quantized PyTorch graph directly often faces compatibility issues because integer operations do not always map cleanly to standard ONNX operators. By exporting the FP32 model first and quantizing the ONNX graph directly, the resulting model can be reliably executed on any device supporting ONNXruntime. Additionally, if this approach were applied to a language model, executing the `export_llm_onnx.py` script as a batch job on the DAS-5 cluster using `torch.onnx.export` would only trace a single forward pass without a KV-cache, making native ONNX quantization routes even more critical for scaling and performance.

(3) Going from FP32 to INT8 transforms a continuous floating-point number into a distinct integer 'payload' alongside specific quantization metadata: a 'scale' factor (defining the step size between integers) and a 'zero-point' (the integer that corresponds to the real value 0.0). This directly reduces the storage requirement per weight from 4 bytes down to 1 byte.

 Latency is reduced on suitable hardware (like dedicated NPUs) because integer ALUs take up less physical silicon area and consume less power than floating-point ALUs. This allows the hardware to use advanced SIMD (Single Instruction, Multiple Data) instructions to process many INT8 operations in parallel within a single clock cycle, significantly accelerating inference throughput.

Model size changed from 8.72 MB (fp32) to 9.48 MB (int8)

## Checklist

- [ ] FP32 baseline trained/loaded and measured on CPU.
- [ ] Dynamic PTQ implemented and measured.
- [ ] Static PTQ (FX graph mode) implemented, calibrated, and measured.
- [ ] Summary table across FP32 / dynamic / static.
- [ ] ONNX export of the FP32 model + ONNX Runtime quantization to INT8,
      with file sizes and a numerical sanity check.
- [ ] All reflection questions because learning is important.